# RQ5 — Fairness / Generalization across Regions
*Does performance differ between Bejaia and Sidi-Bel Abbes, and does a model trained on one region generalize to the other?*

**Outputs:** `RQ5_region_fairness.csv`, `RQ5_region_fairness.pdf`

In [ ]:

# ============================================================
# Shared data loading & preprocessing (Algerian Forest Fires)
# ============================================================
import os, glob, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
plt.rcParams.update({
    "figure.dpi": 120, "savefig.dpi": 300,
    "font.size": 11, "axes.titlesize": 13, "axes.labelsize": 11,
    "axes.grid": True, "grid.alpha": 0.3, "figure.autolayout": True,
})

OUTDIR = "/kaggle/working"          # on Kaggle this is the output folder
os.makedirs(OUTDIR, exist_ok=True)

def find_csv():
    """Find the Algerian Forest Fires csv anywhere under /kaggle/input."""
    cands = glob.glob("/kaggle/input/**/*.csv", recursive=True)
    if not cands:                   # local fallback
        cands = glob.glob("**/*.csv", recursive=True)
    # prefer a file whose name mentions 'algerian' or 'forest'
    for c in cands:
        n = os.path.basename(c).lower()
        if "algerian" in n or "forest" in n or "fire" in n:
            return c
    return cands[0]

def load_algerian():
    path = find_csv()
    print("Loading:", path)
    with open(path, "r", encoding="utf-8", errors="ignore") as fh:
        lines = fh.read().splitlines()
    # The raw UCI file mixes a title line, two region headers and a blank row,
    # so we parse it line by line rather than with a fixed-width csv reader.
    rows = [ln.split(",") for ln in lines]
    header = None
    region = "Bejaia"           # first block in the UCI file
    region_switched = False
    records, cols = [], None
    for r in rows:
        cells = [str(x).strip() for x in r]
        joined = " ".join(cells).lower()
        if "temperature" in joined and ("rh" in joined or "ws" in joined):
            cols = [c.strip() for c in cells if c.strip() != ""]
            header = cols
            if records and not region_switched:
                region = "Sidi-Bel Abbes"   # second header => second region
                region_switched = True
            continue
        if all(c == "" for c in cells):
            continue
        if "region" in joined or "dataset" in joined:   # title / region label line
            if "sidi" in joined:
                region = "Sidi-Bel Abbes"; region_switched = True
            continue
        if header is None:
            continue
        data_cells = [c for c in cells if c != ""]
        if len(data_cells) < len(cols):
            continue
        rec = dict(zip(cols, data_cells[:len(cols)]))
        rec["region"] = region
        records.append(rec)
    df = pd.DataFrame.from_records(records)
    return df

df = load_algerian()
df.columns = [c.strip().replace(" ", "_") for c in df.columns]

# Standardise the target column name
target_col = [c for c in df.columns if "class" in c.lower()]
target_col = target_col[0] if target_col else df.columns[-2]
df = df.rename(columns={target_col: "Classes"})

# Clean target -> binary (fire = 1, not fire = 0)
df["Classes"] = (df["Classes"].astype(str).str.strip().str.lower()
                 .str.replace(r"\s+", " ", regex=True))
df = df[df["Classes"].isin(["fire", "not fire"])].copy()
df["target"] = (df["Classes"] == "fire").astype(int)

# Numeric feature columns
FEATURES = ["Temperature", "RH", "Ws", "Rain",
            "FFMC", "DMC", "DC", "ISI", "BUI", "FWI"]
FEATURES = [f for f in FEATURES if f in df.columns]
for c in FEATURES:
    df[c] = pd.to_numeric(df[c], errors="coerce")
df = df.dropna(subset=FEATURES + ["target"]).reset_index(drop=True)

# --- Ensure two regions exist for the fairness / generalization analysis ---
# The Algerian dataset has two regions (first half Bejaia, second half
# Sidi-Bel Abbes). Some cleaned CSVs drop the region title rows, so we recover
# the split: prefer an explicit region column, else split by position.
if df["region"].nunique() < 2:
    cand = [c for c in df.columns
            if c.lower() in ("region", "area", "location") and c != "region"]
    used = False
    for c in cand:
        vals = df[c].astype(str)
        if vals.nunique() == 2:
            mapping = {v: ("Bejaia" if i == 0 else "Sidi-Bel Abbes")
                       for i, v in enumerate(sorted(vals.unique()))}
            df["region"] = vals.map(mapping)
            used = True
            break
    if not used and len(df) >= 20:
        half = len(df) // 2
        df["region"] = ["Bejaia"] * half + ["Sidi-Bel Abbes"] * (len(df) - half)
        print("Region column not found in data -> split by position "
              "(first half = Bejaia, second half = Sidi-Bel Abbes).")

print("Shape:", df.shape, "| Fire:", int(df.target.sum()),
      "| Not fire:", int((1-df.target).sum()))
print("Regions:", df["region"].value_counts().to_dict())
X = df[FEATURES].copy()
y = df["target"].copy()


In [ ]:

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, recall_score

regions = sorted(df["region"].unique())
print("Regions found:", regions)

def fit_eval(train_mask, test_mask, label):
    Xtr, ytr = X[train_mask], y[train_mask]
    Xte, yte = X[test_mask], y[test_mask]
    m = RandomForestClassifier(n_estimators=400, random_state=42).fit(Xtr, ytr)
    p = m.predict(Xte)
    return {"Scenario": label, "n_test": int(test_mask.sum()),
            "Accuracy": accuracy_score(yte, p),
            "F1": f1_score(yte, p, zero_division=0),
            "Recall": recall_score(yte, p, zero_division=0)}

rows = []
# (a) within-region (stratified split inside each region)
for r in regions:
    mask = df["region"] == r
    idx = df[mask].index
    tr, te = train_test_split(idx, test_size=0.3,
                              stratify=y[mask], random_state=42)
    tr_m = df.index.isin(tr); te_m = df.index.isin(te)
    rows.append(fit_eval(tr_m, te_m, f"Within {r}"))
# (b) cross-region transfer
if len(regions) == 2:
    a, b = regions
    rows.append(fit_eval(df["region"] == a, df["region"] == b,
                         f"Train {a} -> Test {b}"))
    rows.append(fit_eval(df["region"] == b, df["region"] == a,
                         f"Train {b} -> Test {a}"))
fair = pd.DataFrame(rows).round(3)
fair.to_csv(f"{OUTDIR}/RQ5_region_fairness.csv", index=False)
print(fair.to_string(index=False))


In [ ]:

fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(fair)); w = 0.27
ax.bar(x - w, fair["Accuracy"], w, label="Accuracy", color="#0072B2")
ax.bar(x,     fair["F1"],       w, label="F1",       color="#009E73")
ax.bar(x + w, fair["Recall"],   w, label="Recall",   color="#E69F00")
ax.set_xticks(x); ax.set_xticklabels(fair["Scenario"], rotation=20, ha="right")
ax.set_ylim(0, 1.05); ax.set_ylabel("Score")
ax.set_title("RQ5: Within-Region vs Cross-Region Performance")
ax.legend(frameon=True)
fig.savefig(f"{OUTDIR}/RQ5_region_fairness.pdf", bbox_inches="tight")
print("Saved RQ5_region_fairness.pdf")
